In [1]:
import os
import glob
import random
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    DATA_DIR = "/home/Joshua/Downloads/leopard_toad_identification/dataset/dataset_reid_crops"
    WEIGHTS_PATH = "/home/Joshua/Downloads/leopard_toad_identification/identification/weights/resnet50_backbone_final.pth"
    IMG_SIZE = 224 * 2
    BATCH_SIZE = 32
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    SEED = 42

# Ensure reproducibility
torch.manual_seed(Config.SEED)
random.seed(Config.SEED)
np.random.seed(Config.SEED)

In [ ]:
# =============================================================================
# TRANSFORMS
# =============================================================================
class ResizeAndPad:
    def __init__(self, target_size, fill=0):
        self.target_size = target_size
        self.fill = fill

    def __call__(self, image):
        w, h = image.size
        scale = self.target_size / max(w, h)
        image = image.resize((int(w * scale), int(h * scale)), Image.Resampling.BICUBIC)
        delta_w, delta_h = self.target_size - image.size[0], self.target_size - image.size[1]
        padding = (delta_w // 2, delta_h // 2, delta_w - (delta_w // 2), delta_h - (delta_h // 2))
        return T.functional.pad(image, padding, fill=self.fill)

class GaussianBlur:
    def __init__(self, sigma=(0.1, 2.0)):
        self.sigma = sigma
    def __call__(self, x):
        sigma = random.uniform(self.sigma[0], self.sigma[1])
        return x.filter(ImageFilter.GaussianBlur(radius=sigma))

def get_aug_transform():
    return T.Compose([
        ResizeAndPad(Config.IMG_SIZE, fill=0),
        T.Pad(padding=int(Config.IMG_SIZE * 0.1)),
        T.RandomCrop(size=Config.IMG_SIZE),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=15),
        T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
        T.RandomGrayscale(p=0.2),
        T.RandomApply([GaussianBlur([0.1, 2.0])], p=0.5),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

In [ ]:
# =============================================================================
# MODEL & FEATURE LOGIC
# =============================================================================
def load_backbone(weights_path):
    model = models.resnet50(weights=None)
    model.fc = nn.Identity()
    state_dict = torch.load(weights_path, map_location=Config.DEVICE)
    model.load_state_dict(state_dict)
    model.to(Config.DEVICE).eval()
    return model

def extract_combined_features(model, image_paths):
    base_transform = T.Compose([
        T.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    aug_transform = get_aug_transform()

    features_list = []
    metadata = [] # Stores (original_path, is_augmented)

    print("Extracting features for originals and augmentations...")
    with torch.no_grad():
        for p in image_paths:
            img = Image.open(p).convert("RGB")
            
            # Original
            orig_t = base_transform(img).unsqueeze(0).to(Config.DEVICE)
            features_list.append(model(orig_t).cpu().numpy())
            metadata.append((p, False))
            
            # Augmented
            aug_t = aug_transform(img).unsqueeze(0).to(Config.DEVICE)
            features_list.append(model(aug_t).cpu().numpy())
            metadata.append((p, True))

    return np.concatenate(features_list, axis=0), metadata

In [ ]:
# =============================================================================
# VISUALIZATION
# =============================================================================
def visualize_retrieval(features, metadata, k=5):
    # Normalize features
    features_norm = features / np.linalg.norm(features, axis=1, keepdims=True)
    
    # Indices of original images
    orig_indices = [i for i, meta in enumerate(metadata) if not meta[1]]
    query_idx = random.choice(orig_indices)
    
    # Find nearest neighbors
    sims = np.dot(features_norm, features_norm[query_idx])
    top_k = np.argsort(sims)[::-1][:k+1]
    
    fig, axes = plt.subplots(1, k+1, figsize=(18, 4))
    
    # Query Image
    q_path = metadata[query_idx][0]
    axes[0].imshow(Image.open(q_path))
    axes[0].set_title("Query (Original)")
    axes[0].axis('off')
    
    # Matches
    aug_trans = get_aug_transform() # For displaying the augmented match
    for i, idx in enumerate(top_k[1:]):
        path, is_aug = metadata[idx]
        img = Image.open(path)
        if is_aug:
            img = aug_trans(img) # Simple visualization hack
            title = f"Augmented Match\nSim: {sims[idx]:.3f}"
        else:
            title = f"Match {i+1}\nSim: {sims[idx]:.3f}"
            
        axes[i+1].imshow(img)
        axes[i+1].set_title(title)
        axes[i+1].axis('off')
    plt.show()

In [ ]:
# =============================================================================
# MAIN
# =============================================================================
def main():
    image_paths = glob.glob(os.path.join(Config.DATA_DIR, "*.jpg"))
    if not image_paths: return
    
    model = load_backbone(Config.WEIGHTS_PATH)
    features, metadata = extract_combined_features(model, image_paths)
    
    print("Running Retrieval Test (Original vs Augmented)...")
    for _ in range(3):
        visualize_retrieval(features, metadata, k=5)

In [ ]:
main()